<a href="https://colab.research.google.com/github/aripenguin/Data698/blob/main/datascraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#libraries & keys
import requests
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')
import os
from pytrends.request import TrendReq
import time
import matplotlib.pyplot as plt
import seaborn as sns

API_KEY = "your-api-key-here"
BASE_URL = "https://api.themoviedb.org/3"

YOUTUBE_API_KEY = "your-api-key-here"

In [ ]:
#find movies through TMDB
def safe_request(url, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 2))
                print(f"Rate limited. Waiting {wait}s…")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"Error ({e}), retrying ({attempt+1}/{max_retries})…")
            time.sleep(2)
    return None

movies = []
for year in range(2021, 2025):

    max_movies_per_year = 500
    movie_count = 0

    for page in tqdm(range(1, 301)):
        url = f"{BASE_URL}/discover/movie"
        params = {
            "api_key": API_KEY,
            "primary_release_year": year,
            "sort_by": "popularity.desc",
            "region": "US",
            "with_origin_country": "US",
            "page": page,
        }

        data = safe_request(url, params)
        if not data or "results" not in data or len(data["results"]) == 0:
            break

        for m in data["results"]:
            if movie_count >= max_movies_per_year:
                break

            movie_id = m["id"]

            #get providers, check and only add movies that have at least one major US provider
            provider_url = f"{BASE_URL}/movie/{movie_id}/watch/providers"
            provider_data = safe_request(provider_url, {"api_key": API_KEY})

            us = provider_data.get("results", {}).get("US", {}) if provider_data else {}
            flatrate = us.get("flatrate", [])

            movie_providers = [
                p["provider_name"]
                for p in flatrate
                if p["provider_name"] in {
                    "Netflix", "Hulu", "Amazon Prime Video", "Disney Plus",
                    "Max", "HBO Max", "Peacock", "Paramount+", "Apple TV+"
                }
            ]

            if not movie_providers:
                continue

            #for has_theatrical_run (US type == 3, yes)
            release_url = f"{BASE_URL}/movie/{movie_id}/release_dates"
            release_data = safe_request(release_url, {"api_key": API_KEY})

            has_theatrical_release = False
            if release_data:
                for country in release_data.get("results", []):
                    if country.get("iso_3166_1") == "US":
                        for r in country.get("release_dates", []):
                            if r.get("type") == 3:
                                has_theatrical_release = True
                                break

            movies.append({
                "movie_id": movie_id,
                "title": m["title"],
                "release_date": m.get("release_date"),
                "popularity": m.get("popularity"),
                "vote_average": m.get("vote_average"),
                "vote_count": m.get("vote_count"),
                "has_theatrical_release": has_theatrical_release,
                "providers": ", ".join(movie_providers)
            })

            movie_count += 1

        if movie_count >= max_movies_per_year:
            break

df = pd.DataFrame(movies)
df = df.drop_duplicates(subset=["movie_id"]).reset_index(drop=True)

print(f"/nTotal movies fetched: {len(df)}")

In [ ]:
#new additional columns
df["director"] = "N/A"
df["genres"] = "N/A"
df["budget"] = None
df["main_cast"] = "N/A"

df["runtime"] = None
df["production_company_count"] = 0
df["has_major_studio"] = False
df["is_franchise"] = False

MAJOR_STUDIOS = {"Warner Bros.", "Universal Pictures", "Walt Disney Pictures",
    "Paramount Pictures", "Sony Pictures", "20th Century Studios"}

for index, row in tqdm(df.iterrows(), total=len(df)):
    movie_id = row["movie_id"]

    movie_url = f"{BASE_URL}/movie/{movie_id}"
    credits_url = f"{BASE_URL}/movie/{movie_id}/credits"

    movie_data = safe_request(movie_url, {"api_key": API_KEY})
    credits_data = safe_request(credits_url, {"api_key": API_KEY})

    if not movie_data or not credits_data:
        continue

    #director
    for crew_member in credits_data.get("crew", []):
        if crew_member.get("job") == "Director":
            df.at[index, "director"] = crew_member.get("name", "N/A")
            break

    #genres
    movie_genres = [g["name"] for g in movie_data.get("genres", [])]
    if movie_genres:
        df.at[index, "genres"] = ", ".join(movie_genres)

    #budget
    df.at[index, "budget"] = movie_data.get("budget", None)

    #runtime
    if movie_data.get("runtime"):
        df.at[index, "runtime"] = movie_data["runtime"]

    #production_company_count
    companies = movie_data.get("production_companies", [])
    if companies:
        df.at[index, "production_company_count"] = len(companies)

        for c in companies:
            if c.get("name") in MAJOR_STUDIOS:
                df.at[index, "has_major_studio"] = True
                break

    #is_franchise
    if movie_data.get("belongs_to_collection"):
        df.at[index, "is_franchise"] = True

    #main_cast
    cast = [c["name"] for c in credits_data.get("cast", [])[:3]]
    if cast:
        df.at[index, "main_cast"] = ", ".join(cast)


In [ ]:
#more new columns

#release_year separate
df["release_year"] = pd.to_datetime(df["release_date"]).dt.year
#release_month separate
df["release_month"] = pd.to_datetime(df["release_date"]).dt.month
#is_summer_release & is_holiday_release boolean
df["is_summer_release"] = False
df["is_holiday_release"] = False

for i, month in enumerate(df["release_month"]):
    if month in [5, 6, 7, 8]:
        df.at[i, "is_summer_release"] = True

    if month in [11, 12]:
        df.at[i, "is_holiday_release"] = True

#providers -> lists
df["providers"] = df["providers"].apply(lambda x: [p.strip() for p in x.split(",")])

#genres -> lists
df["genres"] = df["genres"].apply(
    lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str)
    else x if isinstance(x, list)
    else []
)

#cast -> lists
df["main_cast"] = df["main_cast"].apply(
    lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str)
    else x if isinstance(x, list)
    else []
)
#store all unique cast
all_cast = set(
    name
    for cast_list in df["main_cast"]
    for name in cast_list
)
print(len(all_cast))

#store all unique directors
unique_directors = set(d for d in df["director"] if d != "N/A")
print(len(unique_directors))

In [ ]:
#cast list dataset
cast_df = pd.DataFrame({"name": list(all_cast)})
cast_df["person_id"] = None
cast_df["popularity"] = None

#get person id from names in cast_list
for i, row in tqdm(cast_df.iterrows(), total=len(cast_df)):
    name = row["name"]

    search_url = f"{BASE_URL}/search/person"
    data = safe_request(search_url, {
        "api_key": API_KEY,
        "query": name
    })

    if data and data.get("results"):
        cast_df.at[i, "person_id"] = data["results"][0]["id"]

#get popularity score from person id
for i, row in tqdm(cast_df.iterrows(), total=len(cast_df)):
    pid = row["person_id"]
    if not pid:
        continue

    person_url = f"{BASE_URL}/person/{pid}"
    person_data = safe_request(person_url, {"api_key": API_KEY})

    if person_data:
        cast_df.at[i, "popularity"] = person_data.get("popularity")


In [ ]:
#director dataset
director_df = pd.DataFrame({"name": list(unique_directors)})
director_df["person_id"] = None
director_df["popularity"] = None

#get person id from names in unique_directors
for i, row in tqdm(director_df.iterrows(), total=len(director_df)):
    name = row["name"]

    search_url = f"{BASE_URL}/search/person"
    data = safe_request(search_url, {
        "api_key": API_KEY,
        "query": name
    })

    if data and data.get("results"):
        director_df.at[i, "person_id"] = data["results"][0]["id"]

#get popularity score from person id
for i, row in tqdm(director_df.iterrows(), total=len(director_df)):
    pid = row["person_id"]
    if not pid:
        continue

    person_url = f"{BASE_URL}/person/{pid}"
    person_data = safe_request(person_url, {"api_key": API_KEY})

    if person_data:
        director_df.at[i, "popularity"] = person_data.get("popularity")

In [ ]:
#save cast_list & directors dataset
cast_path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_cast_list.csv'
)
director_path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_directors.csv'
)

cast_df.to_csv(cast_path, index=False)
director_df.to_csv(director_path, index=False)

In [ ]:
#use cast_list & directors dataset for movies_df

cast_pop_lookup = dict(zip(cast_df["name"], cast_df["popularity"]))
df["cast_star_power"] = df["main_cast"].apply(lambda cast: max([cast_pop_lookup.get(name) or 0 for name in cast], default=0))

director_pop_lookup = dict(zip(director_df["name"], director_df["popularity"]))
df["director_popularity"] = df["director"].map(director_pop_lookup).fillna(0)

In [ ]:
path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_movies_2021_2024-2000.csv'
)

df.to_csv(path, index=False)

In [ ]:
path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_movies_2021_2024-2000.csv'
)

df = pd.read_csv(path)

df.columns

In [ ]:
#success indictator: Google Trends

pytrends = TrendReq()

def get_gt(movie_title, release_date):
    start = pd.to_datetime(release_date)
    end = start + pd.Timedelta(days=28)

    timeframe = f"{start.date()} {end.date()}"

    #common named movies will kill this if we dont add it i.e. It
    query = f"{movie_title} movie"

    pytrends.build_payload([query], timeframe=timeframe)
    df = pytrends.interest_over_time()

    if df.empty:
        return None

    s = df[query]

    peak = s.max()
    auc = s.sum()
    decay = s.iloc[-1] / peak if peak != 0 else 0

    return peak, auc, decay

In [ ]:
#success indictator: Google Trends - continuing the list cause it got cut off

try:
    done = pd.read_csv("movies_checkpoint.csv")
    df.update(done)
except:
    pass

pytrends = TrendReq()

def get_gt(movie_title, release_date):
    start = pd.to_datetime(release_date)
    end = start + pd.Timedelta(days=28)

    timeframe = f"{start.date()} {end.date()}"

    #common named movies will kill this if we dont add it i.e. It
    query = f"{movie_title} movie"

    pytrends.build_payload([query], timeframe=timeframe)
    df = pytrends.interest_over_time()

    if df.empty:
        return None

    s = df[query]

    peak = s.max()
    auc = s.sum()
    decay = s.iloc[-1] / peak if peak != 0 else 0

    return peak, auc, decay

#create columns if needed - first run
for c in ["gt_peak","gt_auc","gt_decay"]:
    if c not in df.columns:
        df[c] = None

for i, row in df.iterrows():

    #skip done rows
    if pd.notna(row["gt_peak"]):
        continue

    title = row["title"]
    date = row["release_date"]

    print(i, title)

    try:
        gt = get_gt(title, date)
        if gt:
            df.loc[i, ["gt_peak","gt_auc","gt_decay"]] = gt
    except:
        pass

    #save movies_checkpoint every 2 movies
    if i % 2 == 0:
        df.to_csv("movies_checkpoint.csv", index=False)

    #slow down to avoid 429 too many requests
    time.sleep(10)


In [ ]:
#success indictator: Youtube

def get_youtube_trailer_stats(movie_title):
    query = f"{movie_title} official trailer"

    url = "https://www.googleapis.com/youtube/v3/search"

    params = {
        "part": "snippet",
        "q": query,
        "key": YOUTUBE_API_KEY,
        "maxResults": 1,
        "type": "video"
    }

    r = requests.get(url, params=params)

    if r.status_code != 200:
        return None

    items = r.json().get("items", [])
    if not items:
        return None

    video_id = items[0]["id"]["videoId"]
    url2 = "https://www.googleapis.com/youtube/v3/videos"

    params2 = {
        "part": "statistics",
        "id": video_id,
        "key": YOUTUBE_API_KEY
    }

    r2 = requests.get(url2, params=params2)

    if r2.status_code != 200:
        return None

    stats = r2.json().get("items", [])
    if not stats:
        return None

    s = stats[0]["statistics"]

    views = int(s.get("viewCount", 0))
    likes = int(s.get("likeCount", 0))

    return views, likes

if "yt_views" not in df.columns:
    df["yt_views"] = None
    df["yt_likes"] = None


for i, row in df.iterrows():

    #avoid done rows
    if pd.notna(row["yt_views"]):
        continue

    title = row["title"]

    print(f"\nProcessing {i}: {title}")

    try:
        result = get_youtube_trailer_stats(title)

        if result:
            views, likes = result
            df.loc[i, ["yt_views", "yt_likes"]] = result
            print(views, likes)
        else:
            print("no data")

    except Exception as e:
        print("Error:", e)

    #save youtube_checkpoint every 5 rows
    if i % 5 == 0:
        df.to_csv("youtube_checkpoint.csv", index=False)
        print("saved checkpoint")

    time.sleep(1)

df.to_csv("youtube_final.csv", index=False)
print("DONE")


In [ ]:
#df[["gt_peak", "gt_auc", "gt_decay"]] = df[["gt_peak", "gt_auc", "gt_decay"]].fillna(0)
final_path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_movies_2021_2024_full.csv'
)

df.to_csv(final_path, index=False)
df

In [ ]:
path = os.path.join(
    '/content/drive/MyDrive',
    'Colab Notebooks/',
    'Data 698',
    'us_movies_2021_2024_full.csv'
)

df = pd.read_csv(path)

df.columns

In [ ]:
#fixing features
#log-scaled features for those w/ a wide range of possible values
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").fillna(0)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").fillna(0)

df["log_budget"] = np.log1p(df["budget"])
df["log_votes"] = np.log1p(df["vote_count"])

# raw votes is unbalanced as 2021 movies has had more time to get votes than 2024 movies
df["vote_density"] = df["vote_count"] / (2025 - df["release_year"] + 1)
df["yt_views_density"] = df["yt_views"] / (2025 - df["release_year"] + 1)
df["yt_likes_density"] = df["yt_likes"] / (2025 - df["release_year"] + 1)

# Netflix originals, Prime exclusives (has_tr = False, provider_count = 1)?

In [ ]:
# Columns to plot
cols = [
    "popularity",
    "vote_average",
    "vote_count",
    "vote_density",
    "gt_peak",
    "gt_auc",
    "gt_decay",
    "yt_views",
    "yt_likes",
    "yt_views_density",
    "yt_likes_density"
]

sns.set(style="whitegrid")

fig, axes = plt.subplots(6, 2, figsize=(10, 10))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.boxplot(x=df[col], ax=axes[i], color="skyblue")
    axes[i].set_title(col)
    axes[i].set_ylabel("")
    axes[i].set_xlabel("Value")

plt.tight_layout()
plt.show()

print(df[cols].describe())

In [ ]:
#success boolean column - ver 2.0 in my models.ipynb
popularity_mean = df['popularity'].mean()
vote_count_density_mean = df['vote_density'].mean()
gt_auc_mean = df['gt_auc'].mean()
yt_views_density_mean = df['yt_views_density'].mean()

conditions = (
    (df['popularity'] >= popularity_mean).astype(int) +
    (df['vote_density'] >= vote_count_density_mean).astype(int) +
    (df['gt_auc'] >= gt_auc_mean).astype(int) +
    (df['yt_views_density'] >= yt_views_density_mean).astype(int)
)

#true if at least 2 of the 4 conditions are met
df['success_bool'] = conditions >= 3

print(df['success_bool'].value_counts())